In [1]:
%pip install -e ./orchestrator
! ansible-galaxy collection install -r orchestrator/ansible/requirements.yaml

import orchestrator
import shutil
assert shutil.which("ansible-playbook")
print(orchestrator.__file__)

Obtaining file:///Users/yassiramami/Downloads/temp_5gbp/orchestrator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for orchestrator (pyproject.toml) ... done
  Created wheel for orchestrator: filename=orchestrator-0.0.1-py3-none-any.whl size=2360 sha256=8a1ebd58f7ea3c2019e4d7f9aa72210ac651b75b8475384b74051959c9e0c446
  Stored in directory: /private/var/folders/wn/8gl78s7j64vdbp1z9b717xzr0000gn/T/pip-ephem-wheel-cache-vwsypqoj/wheels/bd/ba/57/a96b475c1612822303f6f950db393d09764dd5201fc8f34a61
Successfully built orchestrator
  Attempting uninstall: orchestrator
    Found existing installation: orchestrator 0.0.1
    Uninstalling orchestrator-0.0.1:
      Successfully uninstalled orchestrator-0.0.1

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from orchestrator.models import MetaModel

# Choose exactly one input model:
RIC_METAMODEL_FILE = "metamodel_flexric.yaml"  # FlexRIC
# RIC_METAMODEL_FILE = "metamodel_osc_ric.yaml"  # O-RAN SC

metamodel = MetaModel.from_yaml_file(RIC_METAMODEL_FILE)

In [3]:
from orchestrator.models import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "params": {
            "provisioning": {
                "experiment": "ric", # slices experiment
                "extra_path": "/Users/yassiramami/slices/bin",
            },
            "ansible": {
                "roles_path": "/Users/yassiramami/Downloads/temp_5gbp/orchestrator/ansible/roles",
            }
        }
    })
deployment_model.to_yaml_file("deployment_model_1.yaml")
deployment_model.to_yaml_file("deployment_model_full_1.yaml", exclude_defaults=False)

In [4]:
from orchestrator.deployment import BaseBlueprintManagement

reconciled_metamodel, reconciled_deployment_model = BaseBlueprintManagement(metamodel=metamodel, deployment_model=deployment_model).output()

In [5]:
reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_1.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_1.yaml")

In [6]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [7]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/node/bi/provision : Select project 'post5g-beta'] ***********

TASK [provisioning/node/bi/provision : Select project 'post5g-beta'] ***********

In [8]:
metamodel, deployment_model = prov.output()

deployment_model.to_yaml_file("deployment_model_2.yaml")
metamodel.to_yaml_file("metamodel_2.yaml")

# checkpoint 1: provisioned

In [9]:
from orchestrator.deployment import ProvisioningManagement
from orchestrator.models import DeploymentModel, MetaModel
from orchestrator.models.meta.ric import RICComponent, RICComponentType, RICImplementation

deployment_model = DeploymentModel.from_yaml_file("deployment_model_2.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_2.yaml")

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [10]:
prov.get_merged(output="yaml", path="merged_model.yaml")

In [11]:
prov.get_ansible_inventory(path="inventory.ini")

In [12]:
from orchestrator.deployment import K8SSetupManagement

k8s = K8SSetupManagement(metamodel=metamodel, deployment_model=deployment_model)

In [13]:
await k8s.deploy()


PLAY [node2] *******************************************************************

PLAY [node1] *******************************************************************

PLAY [node3] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************
TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************


TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1][WARNING]: Host 'node2' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different int

In [14]:
metamodel, deployment_model = k8s.output()

deployment_model.to_yaml_file("deployment_model_3.yaml")
metamodel.to_yaml_file("metamodel_3.yaml")

# checkpoint 2: k8s ready

In [15]:
from orchestrator.deployment import CoreManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_3.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_3.yaml")

core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

In [16]:
reconciled_metamodel, reconciled_deployment_model = core.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_2.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_2.yaml")

In [17]:
core.mutatable_metamodel.to_yaml_file("mutatated_metamodel.yaml", exclude_defaults=False)

In [18]:
await core.deploy()


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************
TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************
ok: [node1]

PLAY [node1] *******************************************************************
ok: [node1]
TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

TASK [helm/prepare : Select official O-RAN SC M-release chart] ***********

In [19]:
metamodel, deployment_model = core.output()

deployment_model.to_yaml_file("deployment_model_4.yaml")
metamodel.to_yaml_file("metamodel_4.yaml")

# checkpoint 3: core deployed

In [20]:
from orchestrator.deployment import RICManagement

# Deploys the RIC declared in the active metamodel:
# implementation: oai for FlexRIC, or oran_sc for O-RAN SC.
ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)
await ric.deploy()
metamodel, deployment_model = ric.output()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]

TASK [helm/prepare : Select Helm chart source] *********************************
ok: [node1]

TASK [helm/prepare : Remove stale 'oai-flexric-ric' Helm build directory] ******
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Create Helm build directory for 'oai-flexric-ric' chart] ***
changed: [node1]

TASK [helm/prepare : Copy local 'oai-ric' Helm chart directory for 'oai-flexric-ric'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-flexric-ric' Helm Chart.yaml file from Jinja template] ***
changed: [node1]


# checkpoint 4: ric deployed

In [21]:
from orchestrator.deployment import RANManagement

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [22]:
reconciled_metamodel, reconciled_deployment_model = ran.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_3.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_3.yaml")

In [23]:
await ran.remove(force=True)


PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oai-gnb' Helm chart] *************************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/uninstall : Wait for 'oai-gnb' pod to be deleted] *******************
ok: [node1]

TASK [helm/uninstall : Clean up Helm build directory for 'oai-gnb' chart if desired] ***
skipping: [node1]

PLAY RECAP *********************************************************************
node1                      : ok=2    changed=0    unreachable=0    failed=0    skipped=1    rescued=0    ignored=0   


In [24]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]

TASK [helm/prepare : Select Helm chart source] *********************************
ok: [node1]

TASK [helm/prepare : Remove stale 'oai-gnb' Helm build directory] **************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
changed: [node1]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepa

In [25]:
metamodel, deployment_model = ran.output()

deployment_model.to_yaml_file("deployment_model_5.yaml")
metamodel.to_yaml_file("metamodel_5.yaml")

# checkpoint 5: ran deployed

In [26]:
from orchestrator.deployment import UEManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_5.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_5.yaml")

ue = UEManagement(metamodel=metamodel, deployment_model=deployment_model)

In [27]:
reconciled_metamodel, reconciled_deployment_model = ue.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_4.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_4.yaml")

In [28]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]

TASK [helm/prepare : Select Helm chart source] *********************************
ok: [node1]

TASK [helm/prepare : Remove stale 'oai-sim-ue' Helm build directory] ***********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue' chart] *******
changed: [node1]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-sim-ue' Helm Chart.yaml file from Jinja template] ***
changed: [node1]



In [29]:
metamodel, deployment_model = ue.output()

deployment_model.to_yaml_file("deployment_model_6.yaml")
metamodel.to_yaml_file("metamodel_6.yaml")

In [30]:
# await ue.remove()

In [31]:
# await ran.remove(force=True)

In [32]:
# await core.remove()

In [33]:
# \await prov.unprovision()